# Criteria source: protocol instead of final review (split questions)

**One factor changed relative to [08a_split_questions.ipynb](08a_split_questions.ipynb)** (entry 8): the four criteria sections come from the review's **protocol** instead of the final review. Everything else is identical: four `noul` questions in one call, one per section, same question sentence, `state` = record only (no review title, no research questions), same combinations (product of all four; product without outcomes).

Protocol source ([scripts/fulltext_criteria.py](../../scripts/fulltext_criteria.py), `extract_protocols`): pages saved by hand from the Cochrane Library into `data/raw/protocols/cochrane/`, matched by DOI.
- 18 reviews: the published protocol (2003–2017).
- CD000996, CD001261: no protocol is available online (their first versions, 2000 and 1998, are already reviews, abstract only). The previous review version is used instead (2009 `.pub2`, 2014 `.pub3`), which is what the update screeners worked from.

The CLEF TAR labels were made with the final criteria. A drop with protocol criteria therefore includes the real gap between plan and execution, which a model trained on protocols would also face.

Labels: CLEF TAR abstract-level (`label_abs`) and content-level (`label_content`).

Nothing paid runs unless `RUN_SMOKE`, `RUN_SUBSET` or `RUN_PAID` is set to `True` below.

In [1]:
import json, sys
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "README.md").exists())
sys.path.insert(0, str(ROOT / "src"))
import screening as S

RUN_SMOKE = False    # one paid call
RUN_SUBSET = True    # the 10 reviews with < 1000 records (3,637 records); see the cost estimate
RUN_PAID = False     # all 20 reviews; read the cost estimate first
OUT = S.out_dir("protocol_criteria_split_questions")
FINAL = S.out_dir("fulltext_criteria_split_questions")      # same setup with final-review criteria (all 20 reviews)
REFERENCE = {"baseline": S.out_dir("baseline"), "title_abstract_no_results": S.out_dir("review_title_abstract_no_results")}

## Data and split

In [2]:
records, CRITERIA = S.load_records()
records = S.add_bm25(records, CRITERIA)
PROTO = json.loads((ROOT / "data" / "processed" / "tar2019_intervention_protocol_criteria.json").read_text())
FT = json.loads((ROOT / "data" / "processed" / "tar2019_intervention_fulltext_criteria.json").read_text())
assert set(PROTO) == set(CRITERIA)

SECTIONS = {"studies": "Types of studies", "participants": "Types of participants",
            "interventions": "Types of interventions", "outcomes": "Types of outcome measures"}

def split(criteria):
    """{question key: section text without its heading}, split at the '#### Types of ...' headings."""
    parts, key = {}, None
    for line in criteria.split("\n")[1:]:
        head = line.lstrip("#").strip()
        if line.startswith("#### ") and head in SECTIONS.values():
            key = next(k for k, v in SECTIONS.items() if v == head)
            parts[key] = []
        else:
            parts[key].append(line)
    return {k: "\n".join(v).strip() for k, v in parts.items()}

PARTS = {r: split(v["criteria"]) for r, v in PROTO.items()}
FINAL_PARTS = {r: split(v["criteria"]) for r, v in FT.items()}
assert all(set(p) == set(SECTIONS) and all(p.values()) for p in PARTS.values()), "a protocol lacks a section"

n = records.groupby("review").size()
SUBSET = sorted(n[n < 1000].index)
print("subset:", len(SUBSET), "reviews,", int(n[SUBSET].sum()), "records")
pd.DataFrame({**{"type": {r: v["type"] for r, v in PROTO.items()}, "date": {r: v["date"] for r, v in PROTO.items()}},
              **{f"{k}_protocol/final": {r: f"{len(PARTS[r][k])}/{len(FINAL_PARTS[r][k])}" for r in PROTO} for k in SECTIONS}})

20 reviews, 41996 records, 41183 unique PMIDs
not returned by PubMed: 0 | no abstract: 2183 of 41996


subset: 10 reviews, 3637 records


,type,date,studies_protocol/final,participants_protocol/final,interventions_protocol/final,outcomes_protocol/final
CD006468,Protocol,2007,182/36,319/235,453/285,358/1055
CD010753,Protocol,2013,74/74,549/658,1307/1270,1298/1293
CD011571,Protocol,2015,72/76,97/319,141/151,955/842
CD011977,Protocol,2015,58/55,66/88,324/239,2925/2981
CD012342,Protocol,2016,656/393,716/687,851/921,1791/1653
CD009642,Protocol,2012,230/308,467/520,495/444,688/2522
CD011768,Protocol,2015,66/72,302/359,730/706,1970/2282
CD010038,Protocol,2012,53/231,1312/1325,3465/5832,1404/4640
CD010239,Protocol,2012,176/119,75/75,305/558,1584/1683
CD011140,Protocol,2014,577/589,265/242,134/118,2759/3098


## Decision model input

Same function as the split run with final criteria; only `PARTS` differs.

In [3]:
def build_body(review, title, abstract):
    return {
        "model": S.MODEL,
        "state": {"record": {"title": title, "abstract": abstract}},
        "questions": {
            k: {"type": "noul",
                "instructions": f"The record meets this eligibility criterion of the systematic review ({SECTIONS[k].lower()}): "
                                f"it meets the stated requirements and none of the stated exclusions.\n\n{PARTS[review][k]}"}
            for k in SECTIONS
        },
    }

print(json.dumps(build_body(*records.loc[0, ["review", "title", "abstract"]]), indent=1)[:4000])

{
 "model": "typesafe/jev-1.14",
 "state": {
  "record": {
   "title": "Bronchiectasis Severity Is an Independent Risk Factor for Vascular Disease in a Bronchiectasis Cohort.",
   "abstract": "BACKGROUND: Interest in the association of vascular disease with COPD and pneumonia has increased, but there is a lack of research in this area with patients with bronchiectasis.\nMETHODS: A retrospective study of 400 patients attending a specialist bronchiectasis clinic in NHS Lothian (Edinburgh, UK) between May 2013 and September 2014 was conducted. The study assessed the prevalence of vascular disease (ischemic heart disease, cerebrovascular disease, peripheral vascular disease, and atrial fibrillation). Using multivariable models, independent risk factors were identified for vascular disease that developed following the diagnosis of bronchiectasis.\nRESULTS: The study included 400 patients. There was preexisting vascular disease (ie, before the diagnosis of bronchiectasis) in 44 patients (11%

## Cost estimate (no API call)

In [4]:
est = S.estimate_cost(records, build_body)
print(f"subset ({len(SUBSET)} reviews): ${est.loc[SUBSET, 'usd'].sum():.2f}")

TypeSafe | typesafe/jev-1.14-20261014 | USD per 1M input tokens: 0.041999999999999996 | output: 0


requests: 41996 | mean chars per request: 6527 | max: 17372
estimated input tokens: 89.6M | estimated cost: $3.76
subset (10 reviews): $0.24


## Smoke test (one paid call)

In [5]:
if RUN_SMOKE:
    print(json.dumps(S.call(build_body(*records.loc[0, ["review", "title", "abstract"]])), indent=1))

## Subset (paid)

In [6]:
if RUN_SUBSET:
    S.run_reviews(records, SUBSET, build_body, OUT)

## Full run (paid)

In [7]:
if RUN_PAID:
    S.run_reviews(records, CRITERIA, build_body, OUT)

## Results

Protocol vs final criteria on the same records: the reviews with a cache file for this run (the final-criteria split run covers all 20). LGAR baseline and title + abstract without results for reference. A failed call or a missing answer gives the lowest score (counted below).

In [8]:
done = sorted(p.stem for p in OUT.glob("*.jsonl"))
print(len(done), "of", len(CRITERIA), "reviews run")
sub = records[records.review.isin(done)].copy()
for col, path in REFERENCE.items():
    sub = S.add_scores(sub, path, col)

def load_split(path, prefix):
    rows = []
    for r in done:
        for row in map(json.loads, (path / f"{r}.jsonl").open()):
            ans = row["response"].get("answers", {})
            rows.append({"review": r, "pmid": row["pmid"], **{f"{prefix}{k}": ans.get(k, {}).get("noul") for k in SECTIONS}})
    return pd.DataFrame(rows, columns=["review", "pmid", *[prefix + k for k in SECTIONS]]).drop_duplicates(["review", "pmid"], keep="last")

for prefix, path in (("p_", OUT), ("f_", FINAL)):
    sub = sub.merge(load_split(path, prefix), on=["review", "pmid"], how="left")
    cols = [prefix + k for k in SECTIONS]
    bad = sub[cols].isna().any(axis=1)
    print(prefix, "records with a failed call or a missing answer (ranked last):", int(bad.sum()))
    sub[f"{prefix}all4"] = sub[cols].prod(axis=1).where(~bad, -1.0)
    sub[f"{prefix}no_outcomes"] = sub[cols[:3]].prod(axis=1).where(~bad, -1.0)

NAMES = {"baseline": "LGAR baseline (reference)", "title_abstract_no_results": "title + abstract w/o results (reference)",
         "f_all4": "final criteria, product all 4", "f_no_outcomes": "final criteria, without outcomes",
         "p_all4": "protocol criteria, product all 4", "p_no_outcomes": "protocol criteria, without outcomes"}
results = {NAMES[c]: S.evaluate(sub, c) for c in NAMES}
if done:
    for a in ("all4", "no_outcomes"):
        d = results[NAMES[f"p_{a}"]] - results[NAMES[f"f_{a}"]]
        d["reviews"] = results[NAMES[f"f_{a}"]]["reviews"]
        results[f"protocol - final ({a})"] = d
pd.concat(results)

10 of 20 reviews run
baseline: scored 3637 of 3637 records | model: {'typesafe/jev-1.14-20261014'} | cost: $1.80 | without a score (ranked last): 0


title_abstract_no_results: scored 3637 of 3637 records | model: {'typesafe/jev-1.14-20261014'} | cost: $2.06 | without a score (ranked last): 0
p_ records with a failed call or a missing answer (ranked last): 0
f_ records with a failed call or a missing answer (ranked last): 0


MAP  R@1%  R@5%  \
LGAR baseline (reference)                label_abs      70.7  21.0  60.1   
                                         label_content  77.2  31.4  79.8   
title + abstract w/o results (reference) label_abs      74.4  22.3  60.4   
                                         label_content  85.4  36.0  84.7   
final criteria, product all 4            label_abs      68.5  22.8  55.2   
                                         label_content  71.4  28.4  71.9   
final criteria, without outcomes         label_abs      69.9  23.4  56.8   
                                         label_content  77.1  30.2  80.0   
protocol criteria, product all 4         label_abs      69.4  19.9  56.6   
                                         label_content  69.8  24.7  71.2   
protocol criteria, without outcomes      label_abs      68.5  19.6  55.0   
                                         label_content  70.8  26.9  75.6   
protocol - final (all4)                  label_abs       0.9  -2.9   1.4   
                                         label_content  -1.6  -3.7  -0.7   
protocol - final (no_outcomes)           label_abs      -1.4  -3.8  -1.8   
                                         label_content  -6.3  -3.3  -4.4   

                                                        R@10%  R@20%  R@50%  \
LGAR baseline (reference)                label_abs       73.7   84.8   96.8   
                                         label_content   88.6   94.3   99.2   
title + abstract w/o results (reference) label_abs       75.8   86.8   99.0   
                                         label_content   91.8   95.9   99.1   
final criteria, product all 4            label_abs       70.0   80.0   97.5   
                                         label_content   88.3   97.0   99.7   
final criteria, without outcomes         label_abs       70.7   82.3   99.2   
                                         label_content   88.7   96.4   99.4   
protocol criteria, product all 4         label_abs       69.4   79.5   96.9   
                                         label_content   85.1   89.0   99.7   
protocol criteria, without outcomes      label_abs       70.5   79.7   96.3   
                                         label_content   85.4   89.0   99.7   
protocol - final (all4)                  label_abs       -0.6   -0.5   -0.6   
                                         label_content   -3.2   -8.0    0.0   
protocol - final (no_outcomes)           label_abs       -0.2   -2.6   -2.9   
                                         label_content   -3.3   -7.4    0.3   

                                                        WSS@95%  TNR@95%  \
LGAR baseline (reference)                label_abs         60.5     71.1   
                                         label_content     79.1     88.0   
title + abstract w/o results (reference) label_abs         63.8     74.8   
                                         label_content     83.0     92.0   
final criteria, product all 4            label_abs         61.0     71.9   
                                         label_content     82.5     91.8   
final criteria, without outcomes         label_abs         61.9     72.5   
                                         label_content     82.5     91.6   
protocol criteria, product all 4         label_abs         58.7     69.4   
                                         label_content     80.4     89.6   
protocol criteria, without outcomes      label_abs         60.5     71.2   
                                         label_content     80.9     90.1   
protocol - final (all4)                  label_abs         -2.3     -2.5   
                                         label_content     -2.1     -2.2   
protocol - final (no_outcomes)           label_abs         -1.4     -1.3   
                                         label_content     -1.6     -1.5   

                                                        WSS@100%  reviews  
LGAR baseline (reference)                label_abs 

In [9]:
# per review: protocol vs final, and per-section ablations
per = {}
for r, df in sub.groupby("review"):  # empty until a run has finished
    for label in ("label_abs", "label_content"):
        for c in ("f_no_outcomes", "p_no_outcomes"):
            m = S.metrics(df.sort_values([c, "bm25", "pmid"], ascending=[False, False, True])[label])
            per.setdefault(r, {})[(label, c)] = round(m["MAP"] * 100, 1) if m else None
per = pd.DataFrame(per).T
for label in ("label_abs", "label_content") if len(per) else ():
    d = per[(label, "p_no_outcomes")] - per[(label, "f_no_outcomes")]
    print(f"{label}: protocol vs final (without outcomes): better {(d > 0).sum()} / worse {(d < 0).sum()} of {d.notna().sum()} reviews")
if done:
    sec = {}
    for k in SECTIONS:
        for prefix, name in (("f_", "final"), ("p_", "protocol")):
            ev = S.evaluate(sub, prefix + k)
            sec[(k, name)] = {lab: ev.loc[lab, "MAP"] for lab in ("label_abs", "label_content")}
    display(pd.DataFrame(sec).T.rename(columns=lambda c: f"MAP {c} (section alone)"))
per

label_abs: protocol vs final (without outcomes): better 4 / worse 6 of 10 reviews
label_content: protocol vs final (without outcomes): better 2 / worse 6 of 9 reviews


MAP label_abs (section alone)  \
studies       final                              37.2   
              protocol                           38.6   
participants  final                              40.5   
              protocol                           36.0   
interventions final                              60.7   
              protocol                           63.1   
outcomes      final                              29.6   
              protocol                           33.3   

                        MAP label_content (section alone)  
studies       final                                  45.0  
              protocol                               42.0  
participants  final                                  35.9  
              protocol                               26.1  
interventions final                                  63.8  
              protocol                               65.1  
outcomes      final                                  19.3  
              protocol                               23.5

label_abs               label_content              
         f_no_outcomes p_no_outcomes f_no_outcomes p_no_outcomes
CD000996          79.6          85.2         100.0          97.6
CD001261          56.0          54.4          61.5          58.5
CD004414          57.3          73.0          38.2          53.1
CD007867          56.7          47.3          66.7          47.8
CD010239          92.3          90.6         100.0         100.0
CD011140          89.3          85.4           NaN           NaN
CD011571          75.0          80.9          97.6          93.1
CD011977          80.3          81.9          83.0          83.5
CD012164          47.6          22.4          68.1          25.4
CD012551          64.5          64.3          79.1          78.0

In [10]:
# usage on exactly these records
def usage(path, reviews):
    tok = cost = n = 0
    for r in reviews:
        for row in map(json.loads, (path / f"{r}.jsonl").open()):
            u = row["response"].get("usage", {})
            tok += u.get("input_tokens", 0); cost += u.get("cost") or 0; n += 1
    return {"calls": n, "input_tokens": tok, "usd": round(cost, 3), "tokens_per_call": round(tok / max(n, 1))}

pd.DataFrame({"final criteria": usage(FINAL, done), "protocol criteria": usage(OUT, done)}).T if done else None

,calls,input_tokens,usd,tokens_per_call
final criteria,3637.0,5460474.0,0.229,1501.0
protocol criteria,3637.0,4683479.0,0.197,1288.0
